In [1]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [2]:
def find_project_dir_for_my_mac() -> Path:
    """Gets training data path"""
    target_path = Path(os.getcwd()).parent / 'data' / 'cuad' / 'train_separate_questions.json' #curr dir, go to parent, cd into train
    if not target_path.exists():
        print("Feel free to edit this function, otherwise feel free to let Alec know and he'll fix it")
    return target_path
target_path = find_project_dir_for_my_mac()

In [3]:
def load_cuad_df(path):
    """
    Flatten JSON into a tidy DataFrame.
    """
    with open(path) as f:
        raw = json.load(f)

    df = pd.json_normalize(raw["data"], record_path=["paragraphs"], meta=["title"])
    df = df.reset_index(names="doc_id")
    df = df.explode("qas", ignore_index=True) #why explode, at this point there is one row per paragraph but qas inside that row is a list, cant run explode on a qas list
    #horizontally
    df = pd.concat([df.drop(columns="qas"), pd.json_normalize(df["qas"])], axis=1)

    df["category"] = df["question"].str.extract(r'related to "([^"]+)"') #anything in " ... "  in qestion becomes the category for the row
    df = df.explode("answers", ignore_index=True)
    df = pd.concat(
        [df.drop(columns="answers"),
         pd.json_normalize(df["answers"]).rename(columns={"text": "answer_text"})],
        axis=1,
    )
    return df

df = load_cuad_df(target_path)